# CiteSight — Colab Runbook (free T4, 16GB)

Runs **indexing + eval** for CiteSight (visual RAG over SEC filings) without the Docker stack:
Qdrant runs **embedded** (file-based, `QDRANT_MODE=embedded`), models load **sequentially**
(`SEQUENTIAL_MODELS=true`), and the VLM defaults to **7B + 4-bit** (`VLM_VARIANT=7b`,
`VLM_QUANTIZE_4BIT=true`). The same code runs the full Docker app unchanged — everything here is config.

**Colab free-tier reality check** ⏱️: sessions die after ~90 min idle and ~12 h total.
Every expensive step below is **resumable**: model downloads go to a persistent cache dir (Cell 5),
Qdrant/pages/manifest live on Drive, and `citesight ingest` skips already-indexed filings
(manifest + content-hash dedupe). If you get disconnected: rerun Cells 1–5, then continue where you left off.

Run cells **top to bottom**; each prints `✓` or `✗` with what to fix.

## 1 · GPU check

In [ ]:
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, check=True).stdout.strip()
    name, vram = [s.strip() for s in out.split(",")]
    print(f"✓ GPU: {name} ({vram})")
    if "T4" not in name:
        print(f"  note: tuned for T4 16GB; {name} should also work if VRAM >= 15GB")
except (FileNotFoundError, subprocess.CalledProcessError):
    raise SystemExit(
        "✗ No CUDA GPU in this runtime.\n"
        "  Fix: menu → Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save,\n"
        "  then re-run this cell."
    )

## 2 · Mount Google Drive (persistence for Qdrant data, pages, reports)

In [ ]:
from google.colab import drive
import os
try:
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/citesight", exist_ok=True)
    print("✓ Drive mounted; outputs persist under /content/drive/MyDrive/citesight")
except Exception as exc:
    raise SystemExit(f"✗ Drive mount failed ({exc}). Approve the auth popup and re-run.")

## 3 · Clone repo + install (CUDA torch enforced)

In [ ]:
import os, subprocess, sys

# ── EDIT ME: your repo URL (and branch if not the default) ──────────────
REPO_URL = "https://github.com/msd-arch/citesight.git"  # <-- your repo
BRANCH = ""  # empty = default branch
# ─────────────────────────────────────────────────────────────────────────

if not os.path.isdir("/content/citesight"):
    subprocess.run(["git", "clone", REPO_URL, "/content/citesight"], check=True)
if BRANCH:
    subprocess.run(["git", "-C", "/content/citesight", "checkout", BRANCH], check=True)
%cd /content/citesight

# Colab ships a CUDA torch — install the project WITHOUT letting pip replace it.
# (uv.lock pins a CPU/Windows resolution for the dev machine; on Colab we install
#  by range and keep the preinstalled CUDA wheel. transformers is pinned <4.52 —
#  REQUIRED, see constraints.md: 4.52+ silently drops the ColQwen adapter.)
!pip install -q -e ".[dev,gpu]" 2>&1 | tail -2

# Colab preinstalls `torchcodec` (a VIDEO decoder) built for its newer default
# torch; our pinned set resolves torch to 2.6.0, so torchcodec hard-raises on
# import and breaks qwen-vl-utils. We only decode page IMAGES — remove it.
!pip uninstall -y -q torchcodec 2>/dev/null; echo "✓ torchcodec removed (unused video decoder)"

import torch
if not torch.cuda.is_available():
    print("torch lost CUDA during install — forcing the CUDA wheel...")
    !pip install -q --force-reinstall torch --index-url https://download.pytorch.org/whl/cu124 2>&1 | tail -1
    raise SystemExit(
        "CUDA torch reinstalled — one reload needed: Runtime → Restart session,\n"
        "then re-run Cells 1-3 (fast; everything is cached)."
    )
import transformers
assert transformers.__version__ < "4.52", f"✗ transformers {transformers.__version__} breaks ColQwen (see constraints.md)"
print(f"✓ torch {torch.__version__} CUDA ok | transformers {transformers.__version__} | repo installed")

## 4 · Secrets → `.env` (pasted, never hardcoded)

In [ ]:
from getpass import getpass

groq_key = getpass("GROQ_API_KEY (console.groq.com/keys): ").strip()
gemini_key = getpass("GEMINI_API_KEY (optional, Enter to skip — judge defaults to groq): ").strip()
contact = input("Contact email for SEC EDGAR User-Agent (required by EDGAR): ").strip()
if not groq_key:
    raise SystemExit("✗ GROQ_API_KEY is required (free at console.groq.com). Re-run this cell.")
if "@" not in contact:
    raise SystemExit("✗ EDGAR requires a real contact email in the User-Agent. Re-run this cell.")

env = f'''SEC_USER_AGENT="CiteSight/0.1 (research; contact: {contact})"
GROQ_API_KEY={groq_key}
GEMINI_API_KEY={gemini_key}
LLM_PROVIDER=groq
JUDGE_LLM_PROVIDER={'gemini' if gemini_key else 'groq'}
DEVICE=cuda
QDRANT_MODE=embedded
VLM_VARIANT=7b
VLM_QUANTIZE_4BIT=true
SEQUENTIAL_MODELS=true
TRACING=local
DATA_DIR=/content/drive/MyDrive/citesight/data
'''
# ↑ If the 7B VLM OOMs on your GPU, change VLM_VARIANT=7b to VLM_VARIANT=3b and re-run 4→onward.
with open("/content/citesight/.env", "w") as f:
    f.write(env)
print("✓ .env written (keys not echoed). Judge provider:", "gemini" if gemini_key else "groq")

## 5 · Persistent caches (HF models + embedded Qdrant)

In [ ]:
import os, shutil

# ⚠️ WEIGHT SIZES: ColQwen2.5 ~7GB + Qwen2.5-VL-7B ~16GB + bge models ~5GB ≈ 28GB.
# A FREE Google account has 15GB total Drive quota — that will overflow mid-download.
# PERSIST_HF_CACHE=True  -> cache on Drive (survives reconnects; needs ~30GB free Drive)
# PERSIST_HF_CACHE=False -> cache on fast local disk (re-download ~10 min after reconnect)
PERSIST_HF_CACHE = False  # <-- flip to True only if your Drive has ~30GB free

hf_home = "/content/drive/MyDrive/citesight/hf_cache" if PERSIST_HF_CACHE else "/content/hf_cache"
os.makedirs(hf_home, exist_ok=True)
os.environ["HF_HOME"] = hf_home
with open("/content/citesight/.env", "a") as f:
    f.write(f"\nHF_HOME={hf_home}\n")  # for CLI subprocesses too

free_gb = shutil.disk_usage(os.path.dirname(hf_home) or "/").free / 1e9
print(f"✓ HF cache: {hf_home} ({free_gb:.0f} GB free)")
print("✓ Qdrant (embedded) + pages + manifest + reports: /content/drive/MyDrive/citesight/data")
if PERSIST_HF_CACHE and free_gb < 30:
    print(f"✗ WARNING: only {free_gb:.0f} GB free at the cache location — downloads may fail; consider PERSIST_HF_CACHE=False or VLM_VARIANT=3b")

# Optional: gated/private HF models only. All CiteSight defaults are public — Enter to skip.
from getpass import getpass
tok = getpass("HF token (optional, Enter to skip): ").strip()
if tok:
    from huggingface_hub import login
    login(token=tok)
    print("✓ HF login ok")

## 6 · Canary — prove the models are *correct* on GPU before any long run

Shape checks are not enough: a bad transformers version once loaded ColQwen with dropped
adapter weights and produced correctly-shaped **random** embeddings (constraints.md).
`citesight canary` asserts discriminative behavior for the retriever and a read-a-number
check for the VLM. First run also downloads the weights (~10 min on Colab's pipe).

In [ ]:
import subprocess
r = subprocess.run(["citesight", "canary"], cwd="/content/citesight", text=True,
                   capture_output=True)
print(r.stdout[-2000:], r.stderr[-2000:])
if r.returncode != 0:
    raise SystemExit(
        "✗ Canary FAILED — do NOT proceed to indexing.\n"
        "  Common causes: transformers >=4.52 got installed (re-run Cell 3),\n"
        "  or OOM (set VLM_VARIANT=3b in Cell 4 and re-run 4→6)."
    )
print("✓ canary passed — models loaded on GPU and behaved discriminatively")
# NOTE: SEQUENTIAL_MODELS=true means the canary unloads each model when done, so
# used-memory below shows the *freed* state — that's the residency strategy working.
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 7 · Index — AAPL, MSFT, NVDA (10-K + 10-Q, 4 filings each)

Idempotent: already-indexed filings are skipped (manifest + content hash), so re-running
after a disconnect continues instead of restarting. ~30-60 min first time on a T4.

In [ ]:
import subprocess
for ticker in ("AAPL", "MSFT", "NVDA"):
    print(f"=== {ticker} ===")
    r = subprocess.run(
        ["citesight", "ingest", "--ticker", ticker, "--types", "10-K,10-Q", "--limit", "4"],
        cwd="/content/citesight", text=True, capture_output=True,
    )
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit(f"✗ ingest failed for {ticker} — check SEC_USER_AGENT email (Cell 4) / network, then re-run this cell (it resumes).")
print("✓ ingest complete — corpus status:")
!cd /content/citesight && citesight status

## 8 · Eval — `citesight eval` (CiteSight vs naive text-RAG baseline)

Stages: retrieval metrics (nDCG@5 / Recall@5 / MRR for visual, hybrid, baseline) →
VLM answers + baseline answers → LLM-judge correctness + citation accuracy.
Judge/agent responses are disk-cached (`data/llm_cache`) so re-runs stay in free tiers.

In [ ]:
import subprocess
r = subprocess.run(["citesight", "eval"], cwd="/content/citesight", text=True, capture_output=True)
print(r.stdout[-800:])
if r.returncode != 0:
    print(r.stderr[-2500:])
    raise SystemExit(
        "✗ eval failed. If it's a 429/LlmError: your Groq free tier is briefly exhausted —\n"
        "  wait a minute and re-run (cached responses are reused). If OOM: VLM_VARIANT=3b in Cell 4."
    )
from IPython.display import Markdown, display
display(Markdown(open("/content/citesight/eval/reports/latest.md").read()))
print("✓ full JSON: /content/citesight/eval/reports/latest.json")

## 9 · Save outputs to Drive (timestamped zip)

In [ ]:
import shutil, time, os
stamp = time.strftime("%Y%m%d-%H%M%S")
staging = f"/content/citesight_out_{stamp}"
os.makedirs(staging, exist_ok=True)
shutil.copytree("/content/citesight/eval/reports", f"{staging}/reports", dirs_exist_ok=True)
pages_dir = "/content/drive/MyDrive/citesight/data/pages"
if os.path.isdir(pages_dir):
    shutil.copytree(pages_dir, f"{staging}/pages", dirs_exist_ok=True)
zip_path = shutil.make_archive(
    f"/content/drive/MyDrive/citesight/run_{stamp}", "zip", staging
)
print(f"✓ saved: {zip_path}")
print("  (pages + qdrant + manifest already live on Drive under citesight/data — the zip is the shareable snapshot)")

## 10 · (Optional) Interactive ask — grounded answer + citations inline

Retriever + 4-bit VLM co-resident (~13GB on a T4 — fits; if you OOM'd earlier with 7B,
this cell is where 3B matters).

In [ ]:
import sys
sys.path.insert(0, "/content/citesight/src")
import os; os.chdir("/content/citesight")

from citesight.config.settings import get_settings
from citesight.pipeline import ask
from IPython.display import Image as IPyImage, display

question = input("Question about the indexed filings: ").strip() or \
    "What does NVIDIA say about competition?"
result = ask(question, get_settings(), top_k=3)
print(f"\nAnswer ({result.model_id}):\n  {result.answer}\n")
for c in result.claims:
    print(f"  - {c.text}\n    [{c.doc_id} p.{c.page_num}] \"{c.evidence}\"")
if result.citations:
    print("\nTop cited page image:")
    display(IPyImage(filename=result.citations[0].image_path, width=520))

## What loads the models

| Component | File | Loader | Key env vars |
|---|---|---|---|
| Registry (lazy cache, device/dtype resolution, unload) | `src/citesight/models/registry.py` | `ModelRegistry.get_or_load()` via `get_registry()`; `resolve_device()` / `resolve_dtype()` | `DEVICE` (auto/cuda/mps/cpu), `CPU_DTYPE` |
| Visual retriever (ColQwen2.5, `vidore/colqwen2.5-v0.2`) | `src/citesight/models/retriever.py` | `ColQwenRetriever._load()` (lazy on first `embed_*` call); `.unload()` frees it | `RETRIEVER_MODEL_ID`, `QUANTIZE_4BIT` (CUDA only), `EMBED_BATCH_SIZE` |
| VLM answerer (Qwen2.5-VL) | `src/citesight/models/vlm.py` | `QwenVlAnswerer._load()`; variant map `VARIANT_MODEL_IDS = {3b,7b,32b}`; `.unload()` | `VLM_VARIANT`, `VLM_QUANTIZE_4BIT`, `VLM_BACKEND`, `VLM_MAX_PAGES`, `VLM_MAX_NEW_TOKENS`, `VLM_MAX_VISUAL_TOKENS` |
| Text encoder / reranker (bge-m3 / bge-reranker-v2-m3) | `src/citesight/models/text_retrieval.py` | `TextEncoder._load()` / `Reranker._load()` | `DEVICE` |
| Sequential residency (one big model at a time) | `src/citesight/agent/graph.py` + `src/citesight/eval/runner.py` | calls `.unload()` between stages | `SEQUENTIAL_MODELS` |

**Manual loading outside the notebook** — the single file to look at is
`src/citesight/models/retriever.py` (and `vlm.py`); minimal snippet:

```python
from citesight.config.settings import get_settings
from citesight.models.retriever import ColQwenRetriever
from citesight.models.vlm import QwenVlAnswerer

s = get_settings()                # reads .env: DEVICE, VLM_VARIANT, ...
retriever = ColQwenRetriever(s)   # loads lazily on first call
qvec = retriever.embed_query("total net sales")
retriever.unload()                # free VRAM before the VLM (SEQUENTIAL_MODELS)
vlm = QwenVlAnswerer(s)
# vlm.answer(question, [PIL.Image, ...]) -> {answer, claims, raw}
```

Or from a shell: `citesight canary` loads both and verifies them.

**What's persisted where**
- Model weights: `$HF_HOME/hub/...` — Cell 5 sets `HF_HOME` (Drive if `PERSIST_HF_CACHE=True`, else ephemeral `/content/hf_cache`)
- Embedded Qdrant: `$DATA_DIR/qdrant` → `/content/drive/MyDrive/citesight/data/qdrant` (persisted)
- Page images + text: `$DATA_DIR/pages`, manifest: `$DATA_DIR/manifest.db`, LLM response cache: `$DATA_DIR/llm_cache` (all persisted on Drive)
- Eval reports: `/content/citesight/eval/reports` (ephemeral — Cell 9 zips them to Drive)